# NCKH — kiểm tra môi trường Colab trong VS Code

Chọn **Select Kernel → Colab → Auto Connect** và đăng nhập Google nếu được yêu cầu. Notebook kiểm tra môi trường và source; chưa huấn luyện mô hình hay tạo metric hiệu năng.


In [1]:
import platform
import subprocess
import sys
from pathlib import Path

print('Python:', platform.python_version())
print('Runtime:', sys.executable)
if not Path('/content').is_dir():
    raise RuntimeError('Hãy chọn kernel Colab trước khi chạy notebook này.')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False) if Path('/usr/bin/nvidia-smi').exists() else None
print(result.stdout if result is not None else 'Không có GPU; vẫn chạy được kiểm tra source trên CPU.')


Python: 3.13.16
Runtime: /usr/bin/python3
Không có GPU; vẫn chạy được kiểm tra source trên CPU.


In [2]:
REPO = Path('/content/nckh')
URL = 'https://github.com/NeuroDev204/nckh.git'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'main', URL, str(REPO)], check=True)
elif not (REPO / '.git').is_dir():
    raise RuntimeError(f'{REPO} đã tồn tại nhưng không phải repo Git. Chọn đường dẫn khác.')
result = subprocess.run(['git', '-C', str(REPO), 'status', '--short', '--branch'], capture_output=True, text=True, check=True)
print(result.stdout)
result = subprocess.run(['git', '-C', str(REPO), 'log', '-1', '--oneline'], capture_output=True, text=True, check=True)
print(result.stdout)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{REPO}[demo,test]'], check=True)

# Editable install trong subprocess không cập nhật sys.path của kernel đang chạy.
import importlib

source_dir = REPO / 'src'
if not (source_dir / 'nckh/report_charts.py').is_file():
    raise RuntimeError(f'Thiếu source P3 tại {source_dir}; kiểm tra lại repo runtime.')
if str(source_dir) in sys.path:
    sys.path.remove(str(source_dir))
sys.path.insert(0, str(source_dir))
for module_name in tuple(sys.modules):
    if module_name == 'nckh' or module_name.startswith('nckh.'):
        del sys.modules[module_name]
importlib.invalidate_caches()


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-e', '/content/nckh[demo,test]'], returncode=0)

In [7]:
# Áp dụng bản sửa test độc lập dữ liệu máy cũ, chưa có trên origin/main.
# Fixture chỉ được tạo trong thư mục tạm của pytest.
test_patch = 'diff --git a/tests/test_report_charts.py b/tests/test_report_charts.py\nindex a56bca6..5e3fa41 100644\n--- a/tests/test_report_charts.py\n+++ b/tests/test_report_charts.py\n@@ -162,8 +162,10 @@ def test_p3_report_recomputes_cross_split_duplicates(tmp_path: Path) -> None:\n     assert not tables["quality"].set_index("Kiểm tra").loc["Không trùng hash xuyên split", "Đạt"]\n \n \n-def test_pilot_dashboard_renders_saved_figures(pilot_summary: dict, tmp_path: Path) -> None:\n+def test_pilot_dashboard_renders_saved_figures(pilot_summary: dict, tmp_path: Path, monkeypatch: pytest.MonkeyPatch) -> None:\n     AppTest = pytest.importorskip("streamlit.testing.v1").AppTest\n+    write_p3_artifacts(tmp_path / "data/manifests")\n+    monkeypatch.setenv("NCKH_ROOT", str(tmp_path))\n     path = tmp_path / "summary.json"\n     path.write_text(json.dumps(pilot_summary))\n     script = Path(__file__).resolve().parents[1] / "scripts/streamlit_smoke.py"\n@@ -180,8 +182,10 @@ def test_pilot_dashboard_renders_saved_figures(pilot_summary: dict, tmp_path: Pa\n     assert len(app.get("image")) == 2\n \n \n-def test_pilot_dashboard_rejects_bad_json(tmp_path: Path) -> None:\n+def test_pilot_dashboard_rejects_bad_json(tmp_path: Path, monkeypatch: pytest.MonkeyPatch) -> None:\n     AppTest = pytest.importorskip("streamlit.testing.v1").AppTest\n+    write_p3_artifacts(tmp_path / "data/manifests")\n+    monkeypatch.setenv("NCKH_ROOT", str(tmp_path))\n     path = tmp_path / "summary.json"\n     path.write_text("not json")\n     app = AppTest.from_file(str(Path(__file__).resolve().parents[1] / "scripts/streamlit_smoke.py")).run()\n'
check = subprocess.run(
    ['git', 'apply', '--check', '-'], cwd=REPO,
    input=test_patch, capture_output=True, text=True, check=False,
)
if check.returncode == 0:
    subprocess.run(['git', 'apply', '-'], cwd=REPO, input=test_patch, text=True, check=True)
    print('Đã áp dụng bản sửa hai test dashboard.')
else:
    applied = subprocess.run(
        ['git', 'apply', '--reverse', '--check', '-'], cwd=REPO,
        input=test_patch, capture_output=True, text=True, check=False,
    )
    if applied.returncode != 0:
        raise RuntimeError('Không áp dụng được bản sửa test: ' + check.stderr)
    print('Bản sửa test đã được áp dụng từ trước.')

# Le rendu des sept graphiques peut dépasser trois secondes sur CPU Colab.
timeout_patch = '--- a/tests/test_report_charts.py\n+++ b/tests/test_report_charts.py\n@@ -169,7 +169,7 @@\n     path = tmp_path / "summary.json"\n     path.write_text(json.dumps(pilot_summary))\n     script = Path(__file__).resolve().parents[1] / "scripts/streamlit_smoke.py"\n-    app = AppTest.from_file(str(script)).run()\n+    app = AppTest.from_file(str(script), default_timeout=30).run()\n     app.text_input(key="pilot_summary").set_value(str(path)).run()\n     assert not app.exception\n     assert len(app.get("image")) == 7\n'
check = subprocess.run(
    ['git', 'apply', '--check', '-'], cwd=REPO,
    input=timeout_patch, capture_output=True, text=True, check=False,
)
if check.returncode == 0:
    subprocess.run(['git', 'apply', '-'], cwd=REPO, input=timeout_patch, text=True, check=True)
else:
    applied = subprocess.run(
        ['git', 'apply', '--reverse', '--check', '-'], cwd=REPO,
        input=timeout_patch, capture_output=True, text=True, check=False,
    )
    if applied.returncode != 0:
        raise RuntimeError('Không áp dụng được thời gian chờ dashboard: ' + check.stderr)

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', '-ra'],
    cwd=REPO, capture_output=True, text=True, check=False,
)
output = result.stdout + result.stderr
log_path = REPO / 'colab_pytest.log'
log_path.write_text(output, encoding='utf-8')
print(output)
print('Log:', log_path)
result.check_returncode()


Bản sửa test đã được áp dụng từ trước.
........................................................................ [ 75%]
.......................                                                  [100%]
95 passed in 31.81s

Log: /content/nckh/colab_pytest.log


## Kiểm tra dữ liệu dự án trên Drive

Cell sau yêu cầu quyền Drive. Chỉ kiểm tra các đường dẫn dự án, không giải nén hoặc thay đổi dữ liệu. Chỉ chạy training sau khi xác nhận ảnh, nhãn/mask, split và checkpoint.


In [9]:
from google.colab import drive

drive.mount('/content/drive')
MY_DRIVE = Path('/content/drive/MyDrive')
ROOT = MY_DRIVE / 'NCKH_PanDerm'
print('Thư mục dự án:', ROOT)
if ROOT.is_dir():
    for path in sorted(ROOT.iterdir()):
        print('  ', path.name, '(thư mục)' if path.is_dir() else f'({path.stat().st_size:,} bytes)')
else:
    print('Chưa có thư mục dự án tại đường dẫn này.')

candidates = [
    ROOT / 'data/manifests', ROOT / 'nckh/data/manifests',
    ROOT / 'nckh_data', ROOT / 'data', ROOT / 'ISIC2017', ROOT / 'ISIC2018',
    MY_DRIVE / 'nckh_data', MY_DRIVE / 'nckh_root/data/manifests',
    MY_DRIVE / 'ISIC2017', MY_DRIVE / 'ISIC2018',
]
for path in candidates:
    if path.is_dir():
        children = sorted(path.iterdir())
        print('Tìm thấy:', path, '|', len(children), 'mục trực tiếp')
        for child in children[:12]:
            print('  ', child.name)

checkpoint_dir = ROOT / 'checkpoints'
if checkpoint_dir.is_dir():
    checkpoints = [path for path in checkpoint_dir.iterdir() if path.is_file() and path.suffix in ('.pth', '.pt', '.ckpt')]
    print('Checkpoint:', [(path.name, path.stat().st_size) for path in checkpoints])
for base in (ROOT, MY_DRIVE):
    if base.is_dir():
        archives = [path for path in base.iterdir() if path.is_file() and path.suffix.lower() == '.zip' and (base == ROOT or any(word in path.name.lower() for word in ('data', 'isic', 'panderm', 'nckh')))]
        print('ZIP tại', base, ':', [(path.name, path.stat().st_size) for path in archives])
print('Git chỉ cung cấp source; ảnh ISIC và manifest/checkpoint phải được đưa vào runtime hoặc Drive riêng.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Thư mục dự án: /content/drive/MyDrive/NCKH_PanDerm
   checkpoints (thư mục)
   nckh (thư mục)
   runs (thư mục)
Checkpoint: [('panderm_bb_data6_checkpoint-499.pth', 343298382)]
ZIP tại /content/drive/MyDrive/NCKH_PanDerm : []
ZIP tại /content/drive/MyDrive : []
Git chỉ cung cấp source; ảnh ISIC và manifest/checkpoint phải được đưa vào runtime hoặc Drive riêng.


## Chuẩn bị dữ liệu P3

Chạy các cell dưới đây sau khi mount Drive. Tải khoảng 26,8 GB từ ISIC; ảnh và manifest lưu trên Drive, ZIP tạm nằm trong `/content` và được xóa sau khi giải nén thành công. Bước này dùng CPU, không cần GPU. Dữ liệu test được giữ tách biệt; không dùng để chọn checkpoint.


In [10]:
import shutil

DATA = ROOT / 'nckh_data'
MANIFESTS = ROOT / 'data/manifests'
ZIPS = Path('/content/nckh_zips')
LOGS = ROOT / 'runs/p3-data-preparation'
if not (MY_DRIVE.is_dir() and REPO.is_dir()):
    raise RuntimeError('Chạy cell setup và mount Drive trước.')
for directory in (DATA, MANIFESTS, ZIPS, LOGS):
    directory.mkdir(parents=True, exist_ok=True)
print('Ảnh:', DATA)
print('Manifest:', MANIFESTS)
print('Dung lượng trống runtime (GB):', round(shutil.disk_usage('/content').free / 1e9, 1))

def run_prepare(arguments: list[str], log_name: str) -> None:
    log_path = LOGS / log_name
    with log_path.open('a', encoding='utf-8') as log:
        with subprocess.Popen(
            [sys.executable, '-u', *arguments], cwd=REPO,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        ) as process:
            if process.stdout is None:
                raise RuntimeError('Không mở được output subprocess.')
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
    if returncode:
        raise subprocess.CalledProcessError(returncode, arguments)
    print('Log:', log_path)


Ảnh: /content/drive/MyDrive/NCKH_PanDerm/nckh_data
Manifest: /content/drive/MyDrive/NCKH_PanDerm/data/manifests
Dung lượng trống runtime (GB): 93.5


In [11]:
# ISIC 2018: ảnh + mask của ba split chính thức.
run_prepare([
    'scripts/prepare_isic2018.py', '--zips-dir', str(ZIPS),
    '--data-root', str(DATA), '--delete-zips',
], 'isic2018.log')
run_prepare([
    'scripts/build_manifest.py', '--data-root', str(DATA),
    '--out', str(MANIFESTS / 'isic2018_seg.csv'),
], 'manifest.log')
shutil.copy2(DATA / 'ISIC2018/mask_count.json', MANIFESTS / 'mask_count.json')


Tải ISIC2018_Task1-2_Training_Input.zip (11.17 GB)
  ISIC2018_Task1-2_Training_Input.zip: 52.4 / 11165.4 MB (0.5%)
  ISIC2018_Task1-2_Training_Input.zip: 104.9 / 11165.4 MB (0.9%)
  ISIC2018_Task1-2_Training_Input.zip: 157.3 / 11165.4 MB (1.4%)
  ISIC2018_Task1-2_Training_Input.zip: 209.7 / 11165.4 MB (1.9%)
  ISIC2018_Task1-2_Training_Input.zip: 262.1 / 11165.4 MB (2.3%)
  ISIC2018_Task1-2_Training_Input.zip: 314.6 / 11165.4 MB (2.8%)
  ISIC2018_Task1-2_Training_Input.zip: 367.0 / 11165.4 MB (3.3%)
  ISIC2018_Task1-2_Training_Input.zip: 419.4 / 11165.4 MB (3.8%)
  ISIC2018_Task1-2_Training_Input.zip: 471.9 / 11165.4 MB (4.2%)
  ISIC2018_Task1-2_Training_Input.zip: 524.3 / 11165.4 MB (4.7%)
  ISIC2018_Task1-2_Training_Input.zip: 576.7 / 11165.4 MB (5.2%)
  ISIC2018_Task1-2_Training_Input.zip: 629.1 / 11165.4 MB (5.6%)
  ISIC2018_Task1-2_Training_Input.zip: 681.6 / 11165.4 MB (6.1%)
  ISIC2018_Task1-2_Training_Input.zip: 734.0 / 11165.4 MB (6.6%)
  ISIC2018_Task1-2_Training_Input.zip: 7

PosixPath('/content/drive/MyDrive/NCKH_PanDerm/data/manifests/mask_count.json')

In [12]:
# ISIC 2017: ảnh + nhãn phân loại; giữ CSV eval và trainphase riêng.
run_prepare([
    'scripts/prepare_isic2017_cls.py', '--zips-dir', str(ZIPS),
    '--data-root', str(DATA), '--out-dir', str(MANIFESTS), '--delete-zips',
], 'isic2017.log')


Tải ISIC-2017_Training_Part3_GroundTruth.csv (0.00 GB)
  ISIC-2017_Training_Part3_GroundTruth.csv: 0.0 / 0.0 MB (100.0%)
Tải ISIC-2017_Validation_Part3_GroundTruth.csv (0.00 GB)
  ISIC-2017_Validation_Part3_GroundTruth.csv: 0.0 / 0.0 MB (100.0%)
Tải ISIC-2017_Test_v2_Part3_GroundTruth.csv (0.00 GB)
  ISIC-2017_Test_v2_Part3_GroundTruth.csv: 0.0 / 0.0 MB (100.0%)
Tải ISIC-2017_Training_Data.zip (6.23 GB)
  ISIC-2017_Training_Data.zip: 52.4 / 6229.5 MB (0.8%)
  ISIC-2017_Training_Data.zip: 104.9 / 6229.5 MB (1.7%)
  ISIC-2017_Training_Data.zip: 157.3 / 6229.5 MB (2.5%)
  ISIC-2017_Training_Data.zip: 209.7 / 6229.5 MB (3.4%)
  ISIC-2017_Training_Data.zip: 262.1 / 6229.5 MB (4.2%)
  ISIC-2017_Training_Data.zip: 314.6 / 6229.5 MB (5.0%)
  ISIC-2017_Training_Data.zip: 367.0 / 6229.5 MB (5.9%)
  ISIC-2017_Training_Data.zip: 419.4 / 6229.5 MB (6.7%)
  ISIC-2017_Training_Data.zip: 471.9 / 6229.5 MB (7.6%)
  ISIC-2017_Training_Data.zip: 524.3 / 6229.5 MB (8.4%)
  ISIC-2017_Training_Data.zip: 576

In [15]:
# Editable install trong subprocess không cập nhật sys.path của kernel đang chạy.
import importlib

source_dir = REPO / 'src'
if not (source_dir / 'nckh/report_charts.py').is_file():
    raise RuntimeError(f'Thiếu source P3 tại {source_dir}; kiểm tra lại repo runtime.')
if str(source_dir) in sys.path:
    sys.path.remove(str(source_dir))
sys.path.insert(0, str(source_dir))
for module_name in tuple(sys.modules):
    if module_name == 'nckh' or module_name.startswith('nckh.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

import json
import pandas as pd
from nckh.report_charts import p3_tables

summary, tables = p3_tables(MANIFESTS)
classification = pd.read_csv(MANIFESTS / 'isic2017_cls.csv')
missing = [image for image in classification['image'] if not (DATA / 'ISIC2017' / image).is_file()]
print('Ảnh phân loại bị thiếu:', len(missing))
print('Tóm tắt P3:', summary)
display(tables['quality'])
(LOGS / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
if missing:
    raise RuntimeError(f'Thiếu {len(missing)} ảnh phân loại, ví dụ: {missing[:5]}')
if not summary['quality_ok']:
    raise RuntimeError('P3 còn kiểm tra chưa đạt. Xem bảng quality và log trước khi train.')
print('P3 đạt các kiểm tra manifest. Dữ liệu đã lưu trên Drive; chưa chạy fine-tune.')


Ảnh phân loại bị thiếu: 0
Tóm tắt P3: {'seg_images': 3694, 'seg_masks': 3694, 'cls_images': 2750, 'seg_excluded': 0, 'seg_unreadable': 0, 'seg_size_mismatch': 0, 'seg_duplicates': 0, 'seg_sha256': '8cf5e733a18dfbbe8798f03b4f84354594ec0c10e7f8b5072db2dd65733d09fb', 'checksum_matches': True, 'max_masks_per_image': 1, 'trainphase_test_is_val': True, 'quality_ok': True}


,Kiểm tra,Đạt,Chi tiết
0,SHA-256 manifest ISIC 2018,True,8cf5e733a18dfbbe8798f03b4f84354594ec0c10e7f8b5...
1,Đủ split chính thức,True,ISIC 2018 và ISIC 2017 có train/val/test
2,Ảnh ISIC 2018 đọc được,True,3694/3694 ảnh
3,Kích thước ảnh và mask khớp,True,0 cặp lệch kích thước
4,Không có ảnh bị loại,True,0 ảnh bị loại
5,Mask count khớp manifest,True,tối đa 1 mask/ảnh
6,Bảng trùng khớp manifest,True,0 dòng artifact · 0 dòng tính lại
7,Không trùng hash xuyên split,True,0 dòng trùng chính xác
8,Nhãn ISIC 2017 hợp lệ và ID duy nhất,True,0 nhãn lỗi · 0 ID trùng
9,Trainphase dùng validation thay test,True,150 dòng test mô phỏng · test thật giữ riêng 6...


P3 đạt các kiểm tra manifest. Dữ liệu đã lưu trên Drive; chưa chạy fine-tune.


## Smoke test PanDerm classification trên GPU

Chọn runtime GPU T4 trong Colab. Nếu kernel mới chưa mount Drive, chạy `from google.colab import drive; drive.mount("/content/drive")` trước. Chạy các cell phần này; không cần tải lại dataset. Môi trường model dùng Python 3.10 và Torch 2.4.1 riêng, không thay Torch của kernel. Smoke test kiểm tra pipeline và tốc độ, không đo chất lượng head pretrain.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import json
import subprocess
import sys
from pathlib import Path

if not Path('/usr/bin/nvidia-smi').is_file():
    raise RuntimeError('Runtime chưa có GPU. Chọn GPU T4 rồi chạy lại phần smoke test.')
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True)
print(result.stdout)
ROOT = Path('/content/drive/MyDrive/NCKH_PanDerm')
DATA = ROOT / 'nckh_data'
MANIFESTS = ROOT / 'data/manifests'
CHECKPOINT = ROOT / 'checkpoints/panderm_bb_data6_checkpoint-499.pth'
for path in (CHECKPOINT, MANIFESTS / 'isic2017_cls.csv'):
    if not path.is_file():
        raise FileNotFoundError(f'Thiếu {path}. Mount Drive và kiểm tra P3 trước.')
REPO = Path('/content/nckh')
PANDERM = Path('/content/PanDerm')
for repository, url in ((REPO, 'https://github.com/NeuroDev204/nckh.git'), (PANDERM, 'https://github.com/SiyuanYan1/PanDerm.git')):
    if not repository.exists():
        subprocess.run(['git', 'clone', url, str(repository)], check=True)
    elif not (repository / '.git').is_dir():
        raise RuntimeError(f'{repository} tồn tại nhưng không phải repo Git.')
# Pin commit đã kiểm tra trên máy local, trong checkout tạm của Colab.
subprocess.run(['git', '-C', str(PANDERM), 'checkout', 'fd7a80748ba7fc3e203fed88f909f4689d0d6f24'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
MODEL_PYTHON = Path('/content/venv_nckh_cls/bin/python')
if not MODEL_PYTHON.is_file():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '/content/venv_nckh_cls', '--python', '3.10'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(MODEL_PYTHON), 'torch==2.4.1', 'torchvision==0.19.1', '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(MODEL_PYTHON), '-e', str(REPO), 'timm==0.9.16', 'open_clip_torch==2.32.0'], check=True)


Mon Oct 12 08:37:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

CompletedProcess(args=['/usr/bin/python3', '-m', 'uv', 'pip', 'install', '--python', '/content/venv_nckh_cls/bin/python', '-e', '/content/nckh', 'timm==0.9.16', 'open_clip_torch==2.32.0'], returncode=0)

In [8]:
import shutil
from datetime import datetime, timezone
import pandas as pd

manifest = pd.read_csv(MANIFESTS / 'isic2017_cls.csv')
validation = manifest[manifest['split'] == 'val']
if validation.empty:
    raise RuntimeError('Manifest không có split val.')
selected = validation.sample(n=min(20, len(validation)), random_state=2026)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '-colab-smoke-cls'
LOCAL_RUN = Path('/content') / RUN_ID
SMOKE_IMAGES = LOCAL_RUN / 'images'
SMOKE_IMAGES.mkdir(parents=True, exist_ok=False)
for image in selected['image']:
    source = DATA / 'ISIC2017' / image
    if not source.is_file():
        raise FileNotFoundError(source)
    shutil.copy2(source, SMOKE_IMAGES / source.name)
LOCAL_CHECKPOINT = Path('/content/panderm_bb_data6_checkpoint-499.pth')
shutil.copy2(CHECKPOINT, LOCAL_CHECKPOINT)
selected.to_csv(LOCAL_RUN / 'selected_validation.csv', index=False)
print('Đã chuẩn bị', len(selected), 'ảnh validation; seed=2026.')


Đã chuẩn bị 20 ảnh validation; seed=2026.


In [9]:
OUTPUT = LOCAL_RUN / 'result'
SAVED_RUN = ROOT / 'runs' / RUN_ID
SAVED_RUN.mkdir(parents=True, exist_ok=False)
command = [str(MODEL_PYTHON), '-u', 'scripts/bench_inference.py',
           '--task', 'cls', '--panderm-dir', str(PANDERM / 'classification'),
           '--pretrained', str(LOCAL_CHECKPOINT), '--images', str(SMOKE_IMAGES / '*.jpg'),
           '--n', str(len(selected)), '--warmup', '3', '--out-dir', str(OUTPUT)]
result = subprocess.run(command, cwd=REPO, capture_output=True, text=True, check=False)
log = result.stdout + result.stderr
print(log)
(SAVED_RUN / 'worker.log').write_text(log, encoding='utf-8')
if OUTPUT.exists():
    shutil.copytree(OUTPUT, SAVED_RUN, dirs_exist_ok=True)
shutil.copy2(LOCAL_RUN / 'selected_validation.csv', SAVED_RUN / 'selected_validation.csv')
result.check_returncode()
report = json.loads((SAVED_RUN / 'bench_cls.json').read_text())
if report['device'] != 'cuda':
    raise RuntimeError('Model không chạy trên CUDA. Xem worker.log.')
subprocess.run([str(MODEL_PYTHON), '-m', 'nckh.runcard', str(SAVED_RUN), '--seed', '2026',
                '--input', f'pretrained={CHECKPOINT}', '--input', f'manifest={MANIFESTS / "isic2017_cls.csv"}',
                '--config', 'task=smoke_cls', '--config', f'n={len(selected)}'], cwd=REPO, check=True)
print('Báo cáo:', SAVED_RUN)
print('Đây là smoke test head pretrain; chưa kết luận accuracy/F2.')


{
  "task": "cls",
  "n": 20,
  "device": "cuda",
  "gpu_name": "Tesla T4",
  "ms_per_image_mean": 111.40070109998987,
  "ms_per_image_p50": 95.35844250001446,
  "ms_per_image_p95": 252.92688524993991,
  "peak_vram_mb": 350.5
}
SMOKE TEST OK
/content/venv_nckh_cls/lib/python3.10/site-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3609.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/content/nckh/src/nckh/infer.py:164: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights

## Pilot classification: backward, checkpoint và resume

Chạy sau smoke test đạt, trong cùng runtime GPU. Pilot dùng hai ảnh **train có nhãn thật**, hai bước AdamW, kiểm tra gradient/cập nhật backbone và head, lưu checkpoint rồi khôi phục logits và optimizer trước bước 2. Không dùng test/validation để huấn luyện. Checkpoint pilot chỉ phục vụ kiểm tra kỹ thuật.


In [10]:
# Chạy sau phần smoke classification trong cùng runtime GPU.
import json
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

pilot_source = '"""Two optimizer steps with real train labels, including checkpoint resume."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\n\nfrom nckh.infer import ClsPredictor, cls_preprocess\n\n\ndef run_pilot(model: torch.nn.Module, images: torch.Tensor, labels: torch.Tensor,\n              output: Path) -> dict:\n    keys = ("patch_embed.proj.weight", "head.weight")\n    parameters = dict(model.named_parameters())\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=.05)\n    reports = []\n    for step in (1, 2):\n        before = {key: parameters[key].detach().clone() for key in keys}\n        model.train()\n        optimizer.zero_grad(set_to_none=True)\n        loss = torch.nn.functional.cross_entropy(model(images), labels)\n        if not torch.isfinite(loss):\n            raise RuntimeError("Non-finite training loss")\n        loss.backward()\n        gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)\n        if float(gradient_norm) <= 0:\n            raise RuntimeError("Zero gradient norm")\n        for key in keys:\n            gradient = parameters[key].grad\n            if gradient is None or not torch.isfinite(gradient).all() or not gradient.abs().max() > 0:\n                raise RuntimeError(f"Invalid gradient: {key}")\n        optimizer.step()\n        updates = {key: float((parameters[key].detach() - before[key]).abs().max()) for key in keys}\n        if not all(value > 0 for value in updates.values()):\n            raise RuntimeError(f"Weights did not change: {updates}")\n        optimizer_step = max(int(state["step"]) for state in optimizer.state.values())\n        if optimizer_step != step:\n            raise RuntimeError(f"Unexpected optimizer step: {optimizer_step}")\n        reports.append({"step": step, "loss": float(loss.detach()),\n                        "grad_norm": float(gradient_norm), "updates": updates})\n        model.eval()\n        with torch.no_grad():\n            expected = model(images).detach().cpu()\n        checkpoint = output / f"checkpoint-{step}.pth"\n        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": step}, checkpoint)\n        if step == 1:\n            # Corrupt the head first so the reload check actually tests restoration.\n            with torch.no_grad():\n                parameters["head.weight"].zero_()\n            saved = torch.load(checkpoint, map_location="cpu", weights_only=True)\n            model.load_state_dict(saved["model"], strict=True)\n            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=.05)\n            optimizer.load_state_dict(saved["optimizer"])\n            if max(int(state["step"]) for state in optimizer.state.values()) != 1:\n                raise RuntimeError("Optimizer step was not restored")\n            with torch.no_grad():\n                actual = model(images).detach().cpu()\n            if not torch.allclose(actual, expected, atol=1e-5, rtol=1e-4):\n                raise RuntimeError("Checkpoint logits do not match")\n            reload_error = float((actual - expected).abs().max())\n            del saved\n    return {"status": "PASS", "synthetic_targets": False, "stages": reports,\n            "restored_optimizer_step": 1, "max_abs_logits_error": reload_error}\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--panderm-dir", type=Path, required=True)\n    parser.add_argument("--pretrained", type=Path, required=True)\n    parser.add_argument("--manifest", type=Path, required=True)\n    parser.add_argument("--data-root", type=Path, required=True)\n    parser.add_argument("--out-dir", type=Path, required=True)\n    args = parser.parse_args()\n    if not torch.cuda.is_available():\n        raise RuntimeError("This model pilot requires a GPU")\n    table = pd.read_csv(args.manifest)\n    train = table[table["split"] == "train"]\n    if len(train) < 2:\n        raise ValueError("Need at least two train images")\n    selected = train.sample(n=2, random_state=2026)\n    if not selected["label"].isin([0, 1, 2]).all():\n        raise ValueError("Invalid classification labels")\n    torch.manual_seed(2026)\n    np.random.seed(2026)\n    tensors = []\n    for image in selected["image"]:\n        with Image.open(args.data_root / "ISIC2017" / image) as rgb:\n            tensors.append(cls_preprocess(np.asarray(rgb.convert("RGB"))))\n    images = torch.cat(tensors).to("cuda")\n    labels = torch.tensor(selected["label"].tolist(), device="cuda", dtype=torch.long)\n    args.out_dir.mkdir(parents=True, exist_ok=False)\n    selected.to_csv(args.out_dir / "selected_train.csv", index=False)\n    model = ClsPredictor.from_checkpoint(args.panderm_dir, pretrained_path=args.pretrained, device="cuda").model\n    torch.cuda.reset_peak_memory_stats()\n    report = run_pilot(model, images, labels, args.out_dir)\n    report.update(task="cls", seed=2026, n=2, split="train",\n                  peak_vram_mb=torch.cuda.max_memory_allocated() / 2**20)\n    (args.out_dir / "summary.json").write_text(json.dumps(report, indent=2), encoding="utf-8")\n    print(json.dumps(report, indent=2))\n    print("PILOT CLASSIFICATION OK — technical check, not performance evaluation")\n\n\nif __name__ == "__main__":\n    main()\n'
pilot_path = REPO / 'scripts/pilot_classification.py'
pilot_path.write_text(pilot_source, encoding='utf-8')
PILOT_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '-colab-pilot-cls'
PILOT_LOCAL = Path('/content') / PILOT_ID
PILOT_SAVED = ROOT / 'runs' / PILOT_ID
PILOT_SAVED.mkdir(parents=True, exist_ok=False)
command = [str(MODEL_PYTHON), '-u', str(pilot_path),
           '--panderm-dir', str(PANDERM / 'classification'),
           '--pretrained', str(LOCAL_CHECKPOINT),
           '--manifest', str(MANIFESTS / 'isic2017_cls.csv'),
           '--data-root', str(DATA), '--out-dir', str(PILOT_LOCAL)]
result = subprocess.run(command, cwd=REPO, capture_output=True, text=True, check=False)
log = result.stdout + result.stderr
print(log)
(PILOT_SAVED / 'worker.log').write_text(log, encoding='utf-8')
if PILOT_LOCAL.exists():
    shutil.copytree(PILOT_LOCAL, PILOT_SAVED, dirs_exist_ok=True)
result.check_returncode()
report = json.loads((PILOT_SAVED / 'summary.json').read_text())
assert report['status'] == 'PASS'
subprocess.run([str(MODEL_PYTHON), '-m', 'nckh.runcard', str(PILOT_SAVED), '--seed', '2026',
                '--input', f'pretrained={CHECKPOINT}', '--input', f'manifest={MANIFESTS / "isic2017_cls.csv"}',
                '--config', 'task=pilot_cls', '--config', 'n=2'], cwd=REPO, check=True)
print('Pilot đã lưu:', PILOT_SAVED)
print('Chỉ kiểm tra training/resume với hai ảnh train có nhãn thật; chưa đánh giá hiệu năng.')


{
  "status": "PASS",
  "synthetic_targets": false,
  "stages": [
    {
      "step": 1,
      "loss": 1.0985922813415527,
      "grad_norm": 8.40668773651123,
      "updates": {
        "patch_embed.proj.weight": 9.989738464355469e-05,
        "head.weight": 0.00010000023758038878
      }
    },
    {
      "step": 2,
      "loss": 1.0829384326934814,
      "grad_norm": 9.2716646194458,
      "updates": {
        "patch_embed.proj.weight": 9.968876838684082e-05,
        "head.weight": 0.0001001353157334961
      }
    }
  ],
  "restored_optimizer_step": 1,
  "max_abs_logits_error": 0.0,
  "task": "cls",
  "seed": 2026,
  "n": 2,
  "split": "train",
  "peak_vram_mb": 1667.12890625
}
PILOT CLASSIFICATION OK — technical check, not performance evaluation
/content/venv_nckh_cls/lib/python3.10/site-packages/torch/functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorSh